# 과제 B — 같은 일을 세 가지로 엮고 숫자로 비교하기

`openai_agent_intro.ipynb` 와 `openai_agent_code.ipynb` 를 다 이해했다고 보고 내는 과제입니다.
**새로운 기술은 쓰지 않습니다.** 그 두 노트북에 나온 것만으로 만듭니다.

---

## 1. 무엇을 만드나

보안 공지 한 건을 받아 **보안 공지 카드** 한 장을 만듭니다. 전문가 셋이 나눠 씁니다.

```
             +-- 요약가       --> [요약]
공지 본문 ---+-- 영향도 평가자 --> [영향도]   ==> 카드 한 장
             +-- 조치 작성자   --> [조치]
```

같은 카드를 **세 가지 방식**으로 만들고, **숫자로 비교**합니다.

| 방식 | 흐름을 정하는 주체 | 참고 |
|---|---|---|
| ① 코드로 엮기 | 우리 파이썬 (`asyncio.gather`) | `openai_agent_code` Step 3~7 |
| ② 에이전트를 도구로 | 관리자 LLM (`as_tool`) | Step 8 |
| ③ 핸드오프 | 관리자 LLM (`handoffs`) | Step 9 |

### 데이터

`advisories.json` — 실제 공개 공지 **10건**입니다. 직접 만들 필요 없습니다.

| 필드 | 내용 |
|---|---|
| `no` | 1 ~ 10 |
| `cve_id` | CVE 번호 |
| `label` | 제품 이름 |
| `advisory` | 공지 본문 (영문) |

비교는 **한 건으로** 합니다. 공지를 여러 건 쓰면 "방식 때문인지 공지 때문인지" 알 수 없습니다.
1번(Log4Shell)을 권합니다. 길고 내용이 많아 세 전문가가 각자 쓸 거리가 있습니다.

### 쓰는 API 키

`OPENAI_API_KEY` **하나뿐**입니다. 다른 키는 쓰지 않습니다.

---

## 2. 요구사항

### 필수 1 — 전문가 세 명

```python
intro = """...공통 배경..."""
summarizer   = Agent(name="요약가",       instructions=intro + "...", model=MODEL_NAME)
impact_rater = Agent(name="영향도 평가자", instructions=intro + "...", model=MODEL_NAME)
action_writer= Agent(name="조치 작성자",   instructions=intro + "...", model=MODEL_NAME)
```

- **공통 배경은 변수로 빼고** 세 지시문에 더하세요. 복사해 두면 한 곳만 고쳤을 때 조용히 틀립니다
- 세 전문가는 **같은 모델**을 씁니다. 바꾸는 변수를 하나로 둬야 비교가 성립합니다
- 공통 배경에 **"공지에 없는 것을 지어내지 말 것"** 을 넣으세요

### 필수 2 — 방식 ① 코드로 엮기

`asyncio.gather` 로 셋을 동시에 부르고, **파이썬이** 카드를 조립합니다.

### 필수 3 — 방식 ② 에이전트를 도구로

`as_tool()` 로 전문가를 관리자의 도구로 만듭니다.

- **`tool_name` 은 영문만 됩니다** (`^[a-zA-Z0-9_-]+$`). 한글을 넣으면 `BadRequestError` 가 납니다
- 누가 하는 일인지는 `tool_description` 에 한글로 적으세요

### 필수 4 — 방식 ③ 핸드오프

전문가 도구로 재료를 모은 뒤, **카드 조립은 다른 에이전트에게 넘깁니다.**

- 핸드오프 대상의 `name` 도 **영문**이어야 합니다 (`transfer_to_<이름>` 도구가 자동 생성됨)
- **핸드오프가 정말 일어났는지 확인하세요.** 두 가지를 봐야 합니다

```python
kinds = [item.type for item in result.new_items]
"handoff_output_item" in kinds      # 핸드오프가 발생했나
result.last_agent.name              # 마지막으로 답한 에이전트가 누구인가
```

관리자가 넘기지 않고 혼자 카드를 써 버려도 **결과물은 멀쩡한 카드**입니다.
글만 읽어서는 알 수 없습니다.

### 필수 5 — 재기

실행 한 번마다 아래를 기록하세요.

| 항목 | 어디서 |
|---|---|
| 전문가 호출 횟수 | `result.new_items` 중 `tool_call_item` |
| LLM 호출 횟수 | 아래 힌트 참고 |
| 입력·출력 토큰 | 아래 힌트 참고 |
| 걸린 초 | `time.time()` |
| 카드 글자 수 | `len()` |

> **힌트 — 토큰이 이상하다면 세는 곳을 의심하세요.**
>
> 방식 ② 의 LLM 호출 수가 방식 ① 보다 **적게** 나왔다면 뭔가 잘못된 것입니다.
> 전문가 셋을 부르고 카드까지 썼는데 호출이 더 적을 수는 없습니다.
>
> `as_tool()` 로 부른 전문가는 **자기 안에서 따로 LLM 을 부릅니다.**
> 그 호출이 어디에 기록되는지 찾아보세요. `result` 안에 사용량을 담는 곳이 **두 군데** 있습니다.
>
> 측정 코드가 틀리면, 나오는 표는 모델이 아니라 **내 버그**를 재고 있습니다.

### 필수 6 — 세 번 이상 반복

```python
REPEATS = 3
```

**한 번 돌린 결과로 표를 채우면 안 됩니다.**
방식마다 3회 이상 돌리고, 표에는 **평균과 함께 최소~최대**를 적으세요.

`openai_agent_code` Step 8 은 판매원 도구가 4회 호출된 것을 보고
"매번 다른 것이 정상"이라고 적었습니다. **정말 그런지, 어디가 흔들리는지**를 재는 것이 이 과제입니다.

---

## 3. 꼭 보고 넘어갈 것

### 예상한 곳에서 흔들리지 않을 수 있습니다

"전문가를 3번 부른다"는 지시가 지켜지지 않을 것이라고 예상하기 쉽습니다.
**그런데 지켜질 수도 있습니다.**

그때 "역시 지시문도 믿을 만하다"로 끝내지 마세요. **다른 열을 보세요.**
토큰이, 호출 수가, 시간이 흔들리고 있을 수 있습니다.

> 재기 전에 어디가 흔들릴지 정해 놓고 그것만 보면, **흔들리는 다른 곳을 놓칩니다.**
> 그래서 여러 항목을 한꺼번에 재는 것입니다.

### 속도 비교에는 공정하지 않은 부분이 있습니다

방식 ① 은 `asyncio.gather` 로 셋을 **동시에** 부릅니다. ②·③ 은 관리자가 **차례로** 부릅니다.
그래서 ① 이 빠른 데는 "일이 적어서"와 "동시에 해서"가 섞여 있습니다.

**결론에 이 점을 적으세요.** 재는 것만큼 중요한 것이 **잴 수 없는 부분을 밝히는 것**입니다.

---

## 4. 제출물

| 파일 | 내용 |
|---|---|
| `exercise_b.ipynb` | 코드와 실행 결과 (출력을 지우지 말고 저장) |

노트북 마지막에 **네 문장 이상**으로 답하세요.

1. 세 방식 중 어느 것을 쓰겠는가, 그 근거는 어느 숫자인가
2. "전문가 3회 호출" 이 지켜졌는가. 지켜졌다면 **대신 어디가 흔들렸는가**
3. 토큰 차이는 몇 배였는가. 그 차이는 **어디서** 생기는가
4. 속도 비교에서 공정하지 않은 부분은 무엇인가

---

## 5. 평가 기준 (100점)

| 항목 | 배점 | 내용 |
|---|---|---|
| 세 방식 동작 | 25 | 셋 다 카드를 만들어 낸다 |
| 측정 | 25 | 호출 기록·토큰·시간을 **올바른 곳에서** 뽑았다 |
| 반복 | 20 | 3회 이상 돌리고 최소~최대를 표에 적었다 |
| 핸드오프 확인 | 10 | `handoff_output_item` 과 `last_agent` 를 둘 다 봤다 |
| 결론 | 20 | 숫자를 근거로 고르고, 공정하지 않은 부분을 밝혔다 |

**감점**

- 한 번만 돌리고 결론을 씀 (−20) — 이 과제의 핵심을 안 한 것입니다
- 토큰을 `raw_responses` 로만 셈 (−10) — ②·③ 이 실제보다 싸게 나옵니다
- 핸드오프 발생을 확인하지 않음 (−10)
- 지시문에 절차를 다 써 놓음 (−5) — 그러면 ②·③ 이 ① 을 말로 옮긴 것이 됩니다
- 공지를 여러 건 섞어서 비교 (−5) — 방식 차이인지 공지 차이인지 알 수 없습니다

---

## 6. 더 해 볼 것

1. `REPEATS` 를 10 으로 올려 보세요. 3회에서 본 범위가 유지되나요?
2. 공지를 **짧은 것**(5번 ownCloud, 383자)으로 바꿔 보세요. 방식 간 토큰 차이가 줄어드나요?
3. 방식 ② 의 `task` 에서 **"세 결과가 모두 준비될 때까지"** 한 줄을 빼 보세요. 호출 횟수가 달라지나요?
4. 방식 ② 에 `ModelSettings(tool_choice="required")` 를 걸어 보세요. 무엇이 달라지나요?
5. 전문가를 **다섯 명**으로 늘리면 세 방식의 차이가 커지나요, 줄어드나요?

---

## 참고 — 쓰는 노트북

| 필요한 것 | 노트북 |
|---|---|
| `Agent`, `Runner`, `trace` | `openai_agent_intro` |
| `asyncio.gather` 병렬 | `openai_agent_code` Step 4 |
| `as_tool()`, 도구 이름 규칙 | `openai_agent_code` Step 8 |
| `handoffs`, 이름 규칙 | `openai_agent_code` Step 9 |
| 도구 호출 기록 세기 | `openai_agent_code` 36셀 |
| 세 방식 비교표 | `openai_agent_code` Step 10 |

In [6]:
import os
import json
import time
import smtplib
import requests
import asyncio
from datetime import datetime, timedelta, timezone
from email.message import EmailMessage

from dotenv import load_dotenv
from pydantic import BaseModel
from agents import Agent, Runner, trace, function_tool, ModelSettings

load_dotenv(override=True)

MODEL_NAME = "gpt-5.4-mini"

# NVD API 키 (.env 의 NVD_KEY). 없어도 동작하지만 호출 제한이 10배 빡빡해집니다.
NVD_KEY = os.getenv("NVD_API_KEY")

# 호출 간격: 키가 있으면 30초당 50회, 없으면 30초당 5회
NVD_DELAY = 0.6 if NVD_KEY else 6.0

print(f"NVD API 키: {'사용' if NVD_KEY else '없음 (익명 호출)'} / 호출 간격 {NVD_DELAY}초")

NVD API 키: 사용 / 호출 간격 0.6초


In [7]:
ADVISORIES_PATH = "advisories.json"

def load_advisories():
    """보안 공지 10건을 읽어옵니다."""
    with open(ADVISORIES_PATH, encoding="utf-8") as f:
        return json.load(f)

advisories = load_advisories()

print(f"공지 {len(advisories)}건\n")
print(f"{'no':<4} {'CVE':<18} {'제품':<22} 길이")
print("-" * 60)
for a in advisories:
    print(f"{a['no']:<4} {a['cve_id']:<18} {a['label']:<22} {len(a['advisory'])}자")

공지 10건

no   CVE                제품                     길이
------------------------------------------------------------
1    CVE-2021-44228     Apache Log4j2          2500자
2    CVE-2022-22965     Spring Framework       1830자
3    CVE-2017-5638      Apache Struts 2        1726자
4    CVE-2014-6271      GNU Bash               1011자
5    CVE-2023-49105     ownCloud               383자
6    CVE-2026-48710     Starlette              2500자
7    CVE-2026-59822     LiteLLM                1752자
8    CVE-2026-75650     Adobe Commerce         350자
9    CVE-2026-49869     Kestra                 772자
10   CVE-2026-73570     Zimbra Collaboration   418자


In [8]:
intro = """당신은 보안 공지(advisory)를 분석해 보안 공지 카드를 만드는 팀의 일원입니다.
아래는 하나의 CVE에 대한 영문 보안 공지 본문입니다.

지켜야 할 규칙:
- 공지 본문에 없는 내용은 절대 지어내지 마세요. 공지에 없으면 "공지에 명시되지 않음"이라고 쓰세요.
- 답변은 반드시 한국어로 쓰세요.
- 서론·인사말 없이 본론만, 2~3문장 이내로 간결하게 쓰세요.

"""

summarizer = Agent(
    name="요약가",
    instructions=intro + "당신의 역할: 이 취약점이 '무엇'인지 핵심만 요약하세요. (어떤 제품의 어떤 결함인지, 공격자가 뭘 할 수 있는지)",
    model=MODEL_NAME,
)

impact_rater = Agent(
    name="영향도 평가자",
    instructions=intro + "당신의 역할: 이 취약점이 '얼마나' 위험한지 평가하세요. 영향받는 버전, 공격 조건(인증 필요 여부 등), 심각도를 공지 내용에 근거해 설명하세요.",
    model=MODEL_NAME,
)

action_writer = Agent(
    name="조치 작성자",
    instructions=intro + "당신의 역할: 담당자가 '지금 당장' 할 수 있는 조치를 적으세요. 공지에 패치 버전이나 우회책이 있으면 그것을 쓰고, 없으면 그렇다고 명시하세요.",
    model=MODEL_NAME,
)


## 필수 2 — 방식 ① 코드로 엮기

In [12]:
advisory = next(a for a in advisories if a["no"] == 1)
message = advisory["advisory"]

with trace("평가자 3인 병렬 초안"):
    results = await asyncio.gather(
        Runner.run(summarizer, message),
        Runner.run(impact_rater, message),
        Runner.run(action_writer, message),
    )

drafts = [r.final_output for r in results]
names = ["요약가", "영향도 평가자", "조치 작성자"]

for name, draft in zip(names, drafts):
    print("=" * 66)
    print(name)
    print("=" * 66)
    print(draft)
    print()

요약가
Apache Log4j의 `log4j-core`(2.15.0 이전 버전)에서 JNDI LDAP lookup 처리 결함이 있어, 공격자가 제어하는 로그 메시지나 파라미터가 있으면 원격 코드 실행(RCE)이 발생할 수 있습니다. 이 취약점은 설정, 로그 메시지, 파라미터의 JNDI 기능이 공격자 제어 LDAP 등의 엔드포인트를 충분히 막지 못해 생기며, 일부 경우 로그에 포함된 예외/인증 실패 같은 사용자 입력만으로도 악용될 수 있습니다.

영향도 평가자
영향받는 것은 `log4j-core`의 2.15.0 미만 버전이며, 특히 2.14.1 이하에서는 LDAP 등 공격자 제어 JNDI 엔드포인트를 통한 원격 코드 실행(RCE) 가능성이 있습니다. 공격자는 로그 메시지나 파라미터를 통제할 수 있어야 하며(예: 신뢰되지 않은 입력이 로그에 기록되는 경우), 인증 요구 여부는 공지에 명시되지 않았습니다. 심각도는 원격 코드 실행 가능성 때문에 매우 높습니다.

조치 작성자
지금 당장 `org.apache.logging.log4j:log4j-core`를 2.16.0 이상으로 업그레이드하고, 함께 사용하는 `log4j-api`도 동일 버전으로 맞추세요. 공지에는 2.15.0, 2.12.2, 2.3.1로의 백포트가 언급되지만, 2.15.0은 일부 상황에서 여전히 취약하므로 가능한 경우 2.16.0으로 올리는 것이 권장됩니다.



In [13]:
summary, impact, action = [r.final_output for r in results]

card_by_code = f"""[보안 공지 카드] {advisory['cve_id']} - {advisory['label']}

[요약] {summary}

[영향도] {impact}

[조치] {action}
"""
print(card_by_code)

[보안 공지 카드] CVE-2021-44228 - Apache Log4j2

[요약] Apache Log4j의 `log4j-core`(2.15.0 이전 버전)에서 JNDI LDAP lookup 처리 결함이 있어, 공격자가 제어하는 로그 메시지나 파라미터가 있으면 원격 코드 실행(RCE)이 발생할 수 있습니다. 이 취약점은 설정, 로그 메시지, 파라미터의 JNDI 기능이 공격자 제어 LDAP 등의 엔드포인트를 충분히 막지 못해 생기며, 일부 경우 로그에 포함된 예외/인증 실패 같은 사용자 입력만으로도 악용될 수 있습니다.

[영향도] 영향받는 것은 `log4j-core`의 2.15.0 미만 버전이며, 특히 2.14.1 이하에서는 LDAP 등 공격자 제어 JNDI 엔드포인트를 통한 원격 코드 실행(RCE) 가능성이 있습니다. 공격자는 로그 메시지나 파라미터를 통제할 수 있어야 하며(예: 신뢰되지 않은 입력이 로그에 기록되는 경우), 인증 요구 여부는 공지에 명시되지 않았습니다. 심각도는 원격 코드 실행 가능성 때문에 매우 높습니다.

[조치] 지금 당장 `org.apache.logging.log4j:log4j-core`를 2.16.0 이상으로 업그레이드하고, 함께 사용하는 `log4j-api`도 동일 버전으로 맞추세요. 공지에는 2.15.0, 2.12.2, 2.3.1로의 백포트가 언급되지만, 2.15.0은 일부 상황에서 여전히 취약하므로 가능한 경우 2.16.0으로 올리는 것이 권장됩니다.



## 필수 3 — 방식 ② 에이전트를 도구로
as_tool() 로 전문가를 관리자의 도구로 만듭니다.

tool_name 은 영문만 됩니다 (^[a-zA-Z0-9_-]+$). 한글을 넣으면 BadRequestError 가 납니다
누가 하는 일인지는 tool_description 에 한글로 적으세요

In [28]:
# tool_name은 영문만 가능. 누구인지는 설명(description)에 한글로 적는다.
summarize_tool = summarizer.as_tool(
    tool_name="summarize_advisory",
    tool_description="이 취약점이 '무엇'인지 핵심만 요약하세요. (어떤 제품의 어떤 결함인지, 공격자가 뭘 할 수 있는지)")
rate_impact_tool = impact_rater.as_tool(
    tool_name="rate_impact",
    tool_description="이 취약점이 '얼마나' 위험한지 평가하세요. 영향받는 버전, 공격 조건(인증 필요 여부 등), 심각도를 공지 내용에 근거해 설명하세요.")
write_action_tool = action_writer.as_tool(
    tool_name="write_action",
    tool_description="담당자가 '지금 당장' 할 수 있는 조치를 적으세요. 공지에 패치 버전이나 우회책이 있으면 그것을 쓰고, 없으면 그렇다고 명시하세요.")

card_tools = [summarize_tool, rate_impact_tool, write_action_tool]

for t in card_tools:
    print(f"- {t.name:20} {t.description[:40]}")

- summarize_advisory   이 취약점이 '무엇'인지 핵심만 요약하세요. (어떤 제품의 어떤 결함인지
- rate_impact          이 취약점이 '얼마나' 위험한지 평가하세요. 영향받는 버전, 공격 조건(
- write_action         담당자가 '지금 당장' 할 수 있는 조치를 적으세요. 공지에 패치 버전이


In [29]:
# 2. 관리자 지시문 — "하나만 고른다"가 아니라 "셋을 합친다"
manager_instructions = """당신은 보안 공지 카드를 만드는 관리자입니다.
summarize_advisory, rate_impact, write_action 세 도구를 사용해
보안 공지 하나를 분석하고, 세 결과를 합쳐 보안 공지 카드 한 장을 작성하는 것이 목표입니다.
공지 본문에 없는 내용은 지어내지 마세요.
"""

task = f"""아래 보안 공지를 분석해, 요약/영향도/조치를 모두 담은
보안 공지 카드 한 장을 한국어로 작성하세요.
당신이 가진 도구를 활용해 필요한 정보를 모으세요.
공지에 없는 내용은 지어내지 마세요.

공지 본문:
{advisory['advisory']}
"""

card_manager = Agent(
    name="보안 공지 카드 관리자",
    instructions=manager_instructions,
    tools=card_tools,
    model=MODEL_NAME,
)

print("관리자 준비 완료 — 도구", len(card_manager.tools), "개")

관리자 준비 완료 — 도구 3 개


In [30]:
print("관리자 준비 완료 — 도구", len(card_manager.tools), "개")
with trace("보안 공지 카드 관리자 (에이전트를 도구로)"):
    result = await Runner.run(card_manager, task)

print("\n관리자 최종 응답:", result.final_output)

관리자 준비 완료 — 도구 3 개

관리자 최종 응답: [보안 공지 카드] CVE-2021-44228 - Apache Log4j2

[요약]
Apache Log4j의 `log4j-core`에서 발생한 결함으로, 2.16.0 이전 버전은 LDAP 등 공격자가 제어하는 JNDI 엔드포인트를 통해 원격 코드 실행(RCE)을 허용할 수 있습니다. 공격자는 로그 메시지나 그 파라미터를 조작해 임의 코드를 실행할 수 있으며, `log4j-api`는 직접 영향 대상이 아니지만 함께 같은 버전으로 맞춰야 합니다.

[영향도]
영향받는 버전은 Log4j 2.15.0 미만이며, 특히 `log4j-core` 패키지가 직접 영향받습니다(일부 백포트는 2.3.1, 2.12.2, 2.12.3에 제공됨). 공격자는 인증 없이도 로그 메시지나 파라미터를 통제할 수 있으면 LDAP 등 JNDI 엔드포인트를 통해 원격 코드 실행을 유발할 수 있어 매우 위험합니다. 2.16.0에서 기본 비활성화되며, 2.15.0의 수정은 모든 상황을 막지 못한다고 공지에 명시되어 있습니다.

[조치]
즉시 `org.apache.logging.log4j:log4j-core`를 2.16.0 이상으로 업그레이드하세요(공지에 따르면 2.3.1, 2.12.2, 2.12.3으로의 백포트도 제공됨). `log4j-api`를 함께 사용 중이면 `log4j-core`와 같은 버전으로 맞추고, 2.15.0은 완전한 해결이 아니므로 지금 당장 최종 대응 버전으로 올리는 것이 필요합니다.


In [31]:
calls = [getattr(item.raw_item, "name", "?")
    for item in result.new_items if item.type == "tool_call_item"]

print("도구 호출 순서:")
for i, name in enumerate(calls, 1):
    print(f"  {i}. {name}")

EXPERT_TOOL_NAMES = {"summarize_advisory", "rate_impact", "write_action"}
expert_calls = [c for c in calls if c in EXPERT_TOOL_NAMES]
missing = EXPERT_TOOL_NAMES - set(expert_calls)

print()
print(f"전문가 도구 호출: {len(expert_calls)}회  (지시대로면 3회, 서로 다른 도구 3개)")
print(f"호출 안 된 전문가: {missing if missing else '없음'}")


도구 호출 순서:
  1. summarize_advisory
  2. rate_impact
  3. write_action

전문가 도구 호출: 3회  (지시대로면 3회, 서로 다른 도구 3개)
호출 안 된 전문가: 없음


## 필수 4 — 방식 ③ 핸드오프
전문가 도구로 재료를 모은 뒤, 카드 조립은 다른 에이전트에게 넘깁니다.

핸드오프 대상의 name 도 영문이어야 합니다 (transfer_to_<이름> 도구가 자동 생성됨)
핸드오프가 정말 일어났는지 확인하세요. 두 가지를 봐야 합니다

In [33]:
card_writer_agent = Agent(
    name="CardWriter",
    instructions=intro + (
        "당신의 역할: 지금까지 모인 요약·영향도·조치 내용을 받아, "
        "아래 형식으로 보안 공지 카드 한 장을 작성하세요.\n\n"
        "[보안 공지 카드] <CVE ID> - <제품명>\n\n"
        "[요약]\n...\n\n[영향도]\n...\n\n[조치]\n...\n"
    ),
    model=MODEL_NAME,
)

In [34]:
handoff_instructions = """당신은 보안 공지 카드를 만드는 관리자입니다.
summarize_advisory, rate_impact, write_action 세 도구를 사용해 보안 공지 하나를 분석하세요.
세 결과가 모두 모이면, 카드 작성은 직접 하지 말고 CardWriter 에게 넘기세요.
공지 본문에 없는 내용은 지어내지 마세요.
"""

handoff_task = f"""아래 보안 공지를 분석해, 요약/영향도/조치를 모두 담은
보안 공지 카드 한 장을 한국어로 작성하세요.
당신이 가진 도구를 활용해 필요한 정보를 모으세요.
공지에 없는 내용은 지어내지 마세요.

공지 본문:
{advisory['advisory']}
"""

handoff_manager = Agent(
    name="보안 공지 카드 관리자 (핸드오프형)",
    instructions=handoff_instructions,
    tools=card_tools,
    handoffs=[card_writer_agent],
    model=MODEL_NAME,
)

print("도구", len(handoff_manager.tools), "개 / 핸드오프", len(handoff_manager.handoffs), "개")

도구 3 개 / 핸드오프 1 개


In [35]:
with trace("보안 공지 카드 관리자 (핸드오프)"):
    result3 = await Runner.run(handoff_manager, handoff_task)

print(result3.final_output)

kinds = [item.type for item in result3.new_items]
print("\n핸드오프 발생:", "handoff_output_item" in kinds)
print("마지막으로 답한 에이전트:", result3.last_agent.name)


[보안 공지 카드] CVE-2021-44228 - Apache Log4j

[요약]
Apache Log4j의 `log4j-core`에서 JNDI LDAP 조회 처리를 통해 원격 코드 실행(RCE)이 가능한 취약점입니다. 공격자는 로그 메시지나 파라미터를 제어할 수 있으면, 악의적인 LDAP/JNDI 엔드포인트를 통해 임의 코드를 실행할 수 있습니다.

[영향도]
영향받는 버전은 `log4j-core`의 2.15.0 미만이며, 특히 2.14.1 이하에서는 LDAP 등 공격자가 제어하는 JNDI 엔드포인트를 통해 원격 코드 실행(RCE)이 가능하다고 공지에 명시되어 있습니다. 공격자는 인증 없이도, 로그 메시지나 파라미터처럼 공격자가 제어할 수 있는 데이터를 애플리케이션이 로그에 기록하게 만들 수 있으면 악용할 수 있어 매우 높은 위험입니다.

[조치]
지금 당장 `org.apache.logging.log4j:log4j-core`를 2.16.0 이상으로 업그레이드하고, `log4j-api`도 같은 버전으로 맞추세요. 공지에는 2.16.0이 JNDI를 기본 비활성화하고 메시지 lookups를 완전히 제거한다고 되어 있으며, 그 외 우회책은 공지에 명시되지 않음입니다.

핸드오프 발생: True
마지막으로 답한 에이전트: CardWriter


## 필수 5·6 — 측정 + 반복

세 방식을 함수로 감싸서, 한 번 실행할 때마다 아래를 함께 기록합니다.

- 걸린 시간 (`time.time()`)
- 전문가 호출 횟수 (`tool_call_item` 개수 — 방식①은 코드가 항상 3번 부르도록 짜여 있으므로 고정값)
- LLM 호출 횟수 — `result.context_wrapper.usage.requests`를 씁니다.
  `raw_responses`는 관리자 자신의 호출만 잡고, `as_tool()`로 부른 하위 전문가의 호출은 거기 안 잡힙니다.
  전체 호출 수를 보려면 `context_wrapper.usage`(중첩 호출까지 누적됨)를 써야 합니다.
- 입력·출력 토큰 (`context_wrapper.usage.input_tokens` / `.output_tokens`)
- 카드 글자 수 (`len()`)

방식③은 추가로 핸드오프가 실제로 일어났는지(`handoff_output_item`, `last_agent`)도 매 회 기록합니다.

In [36]:
def count_expert_calls(result):
    calls = [getattr(item.raw_item, "name", "?")
             for item in result.new_items if item.type == "tool_call_item"]
    return len([c for c in calls if c in EXPERT_TOOL_NAMES])


async def run_by_code():
    t0 = time.time()
    results = await asyncio.gather(
        Runner.run(summarizer, message),
        Runner.run(impact_rater, message),
        Runner.run(action_writer, message),
    )
    elapsed = time.time() - t0

    summary_r, impact_r, action_r = [r.final_output for r in results]
    card = f"""[보안 공지 카드] {advisory['cve_id']} - {advisory['label']}

[요약] {summary_r}

[영향도] {impact_r}

[조치] {action_r}
"""
    return {
        "elapsed": elapsed,
        "expert_calls": 3,  # 코드가 항상 3번 부르도록 짜여 있음 (고정값)
        "llm_calls": sum(r.context_wrapper.usage.requests for r in results),
        "input_tokens": sum(r.context_wrapper.usage.input_tokens for r in results),
        "output_tokens": sum(r.context_wrapper.usage.output_tokens for r in results),
        "card_len": len(card),
    }


async def run_as_tool():
    t0 = time.time()
    with trace("방식2 측정"):
        result = await Runner.run(card_manager, task)
    elapsed = time.time() - t0

    usage = result.context_wrapper.usage
    return {
        "elapsed": elapsed,
        "expert_calls": count_expert_calls(result),
        "llm_calls": usage.requests,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "card_len": len(result.final_output),
    }


async def run_handoff():
    t0 = time.time()
    with trace("방식3 측정"):
        result = await Runner.run(handoff_manager, handoff_task)
    elapsed = time.time() - t0

    kinds = [item.type for item in result.new_items]
    usage = result.context_wrapper.usage
    return {
        "elapsed": elapsed,
        "expert_calls": count_expert_calls(result),
        "llm_calls": usage.requests,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "card_len": len(result.final_output),
        "handoff_happened": "handoff_output_item" in kinds,
        "last_agent": result.last_agent.name,
    }


print("측정 함수 준비 완료: run_by_code, run_as_tool, run_handoff")

측정 함수 준비 완료: run_by_code, run_as_tool, run_handoff


In [37]:
REPEATS = 3

rows_code = [await run_by_code() for _ in range(REPEATS)]
print(f"방식① 완료 ({REPEATS}회)")

rows_tool = [await run_as_tool() for _ in range(REPEATS)]
print(f"방식② 완료 ({REPEATS}회)")

rows_handoff = [await run_handoff() for _ in range(REPEATS)]
print(f"방식③ 완료 ({REPEATS}회)")

방식① 완료 (3회)
방식② 완료 (3회)
방식③ 완료 (3회)


In [38]:
def summarize(rows, label):
    metrics = ["elapsed", "expert_calls", "llm_calls", "input_tokens", "output_tokens", "card_len"]
    print(f"=== {label} (n={len(rows)}) ===")
    for m in metrics:
        values = [r[m] for r in rows]
        avg = sum(values) / len(values)
        print(f"  {m:15} 평균 {avg:8.2f}   범위 {min(values)} ~ {max(values)}")
    print()


summarize(rows_code, "① 코드로 엮기")
summarize(rows_tool, "② 에이전트를 도구로")
summarize(rows_handoff, "③ 핸드오프")

print("--- 방식③ 핸드오프 확인 (매 회) ---")
for i, r in enumerate(rows_handoff, 1):
    print(f"  {i}회차: 핸드오프 발생={r['handoff_happened']}  마지막 에이전트={r['last_agent']}")

=== ① 코드로 엮기 (n=3) ===
  elapsed         평균     2.52   범위 2.0273277759552 ~ 3.3550796508789062
  expert_calls    평균     3.00   범위 3 ~ 3
  llm_calls       평균     3.00   범위 3 ~ 3
  input_tokens    평균  2348.00   범위 2348 ~ 2348
  output_tokens   평균   324.67   범위 306 ~ 341
  card_len        평균   652.00   범위 610 ~ 689

=== ② 에이전트를 도구로 (n=3) ===
  elapsed         평균    13.25   범위 13.00915241241455 ~ 13.41474986076355
  expert_calls    평균     3.00   범위 3 ~ 3
  llm_calls       평균     5.00   범위 5 ~ 5
  input_tokens    평균  6698.00   범위 6590 ~ 6753
  output_tokens   평균  2591.33   범위 2505 ~ 2636
  card_len        평균   723.33   범위 679 ~ 765

=== ③ 핸드오프 (n=3) ===
  elapsed         평균    14.42   범위 11.752601623535156 ~ 18.907053232192993
  expert_calls    평균     3.00   범위 3 ~ 3
  llm_calls       평균     6.67   범위 6 ~ 8
  input_tokens    평균 11005.67   범위 9721 ~ 13519
  output_tokens   평균  2214.00   범위 2140 ~ 2336
  card_len        평균   188.00   범위 0 ~ 564

--- 방식③ 핸드오프 확인 (매 회) ---
  1회차: 핸드오프 발생=True  

## 결론

> 아래는 작성 틀입니다. 위 표에서 나온 **실제 숫자**로 채워 완성하세요 — 숫자를 지어내지 마세요.

1. 세 방식 중 **( )** 를 쓰겠다. 근거는 ( ) 표의 ( ) 수치다.
2. "전문가 3회 호출"은 방식②에서 평균 ( )회, 방식③에서 평균 ( )회로 (지켜졌다 / 지켜지지 않았다). 대신 ( ) 항목이 ( )~( ) 범위로 흔들렸다.
3. 토큰 차이는 방식①→②에서 약 ( )배, 방식①→③에서 약 ( )배였다. 이 차이는 ( )에서 생긴다 (예: `as_tool()`로 부른 하위 전문가 호출이 관리자 자신의 호출에 더해 추가로 토큰을 쓰기 때문 등).
4. 속도 비교에서 방식①은 `asyncio.gather`로 셋을 **동시에** 부르고, 방식②·③은 관리자가 **차례로** 부른다. 그래서 속도 차이에는 "일이 적어서"와 "동시에 해서"가 섞여 있어 공정한 비교가 아니다.